# Sprint 9 - Otimização de Modelos

Sprint criada para otimização dos modelos Logistic Regression e XGBoost

---

## Etapa 1 - Importar libs

In [7]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from xgboost import XGBClassifier

---

## Etapa 2 - Carregar dataset

In [2]:
df = pd.read_csv("encoded_df.csv")

df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


---

## Etapa 3 - Separar X e Y

In [3]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

---

## Etapa 4 - Train Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

---

## Etapa 5 - Testar Logistic Regression Balanceada

In [29]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [30]:
lr_balanced = LogisticRegression(
    random_state=42,
    max_iter=5000,
    class_weight="balanced"
)

In [31]:
lr_balanced.fit(
    X_train_scaled,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [33]:
y_pred_lr = lr_balanced.predict(
    X_test_scaled
)

---

## Etapa 6 - Avaliar Logistic Regression Balanceada

In [39]:
y_pred_lr = lr_balanced.predict(X_test_scaled)

accuracy_lr = accuracy_score(y_test, y_pred_lr)

precision_lr = precision_score(y_test, y_pred_lr)

recall_lr = recall_score(y_test, y_pred_lr)

f1_lr = f1_score(y_test, y_pred_lr)

print("Accuracy:", round(accuracy_lr,4))
print("Precision:", round(precision_lr,4))
print("Recall:", round(recall_lr,4))
print("F1:", round(f1_lr,4))

Accuracy: 0.7264
Precision: 0.4909
Recall: 0.7968
F1: 0.6075


In [35]:
print(df["Churn"].unique())

[0 1]


---

## Etapa 7 - XGBoost Tuning

In [40]:
xgb_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_tuned.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


---

## Etapa 8 - Avaliar o XGBoost

In [46]:
y_pred_xgb = xgb_tuned.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

precision_xgb = precision_score(y_test, y_pred_xgb)

recall_xgb = recall_score(y_test, y_pred_xgb)

f1_xgb = f1_score(y_test, y_pred_xgb)

print("Accuracy:", round(accuracy_xgb,4))
print("Precision:", round(precision_xgb,4))
print("Recall:", round(recall_xgb,4))
print("F1:", round(f1_xgb,4))

Accuracy: 0.7875
Precision: 0.6183
Recall: 0.5241
F1: 0.5673


---

## Etapa 9 - Comparação de Modelos

In [42]:
results = pd.DataFrame({
    "Modelo": [
        "Logistic Balanced",
        "XGBoost Tunado"
    ],
    "Accuracy": [
        accuracy_lr,
        accuracy_xgb
    ],
    "Precision": [
        precision_lr,
        precision_xgb
    ],
    "Recall": [
        recall_lr,
        recall_xgb
    ],
    "F1": [
        f1_lr,
        f1_xgb
    ]
})

results.sort_values(
    by="F1",
    ascending=False
)

,Modelo,Accuracy,Precision,Recall,F1
0,Logistic Balanced,0.726368,0.490939,0.796791,0.607543
1,XGBoost Tunado,0.787491,0.618297,0.524064,0.567294


---

## Etapa 10 - Threshold Tuning

In [44]:
probabilities = lr_balanced.predict_proba(X_test_scaled)[:,1]

In [47]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

results_threshold = []

for threshold in thresholds:

    preds = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        preds
    )

    recall = recall_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds
    )

    results_threshold.append([
        threshold,
        precision,
        recall,
        f1
    ])

pd.DataFrame(
    results_threshold,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)

,Threshold,Precision,Recall,F1
0,0.50,0.490939,0.796791,0.607543
1,0.45,0.471212,0.831551,0.601547
2,0.40,0.452150,0.871658,0.595434
3,0.35,0.429299,0.901070,0.581536
4,0.30,0.414576,0.927807,0.573080


---

## Salvar Modelo

In [49]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Escalonamento
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo final
lr_balanced = LogisticRegression(
    random_state=42,
    max_iter=5000,
    class_weight='balanced'
)

lr_balanced.fit(
    X_train_scaled,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [53]:
import joblib

# Modelo
joblib.dump(
    lr_balanced,
    "../models/final_model.pkl"
)

# Scaler
joblib.dump(
    scaler,
    "../models/final_scaler.pkl"
)

# Features
joblib.dump(
    X.columns.tolist(),
    "../models/feature_names.pkl"
)

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


In [54]:
modelo = joblib.load(
    "../models/final_model.pkl"
)

print(type(modelo))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
